# Limpieza y control de calidad 2024-2025

Este notebook audita y genera una copia limpia de las tablas integradas. Conserva las 77 columnas: no elige target ni excluye predictores.

Cambios automáticos permitidos en esta etapa:

- Recortar espacios en variables de texto.
- Convertir literales inequívocos de faltante (`<NA>`, `NA`, `nan`, `null`, texto vacío) a valores faltantes reales.
- Convertir las dos columnas de fecha a tipo fecha cuando sean válidas.
- Convertir el código 99.9 de hemoglobina ajustada a faltante real.

Los valores fuera de rango, códigos especiales y contradicciones lógicas se reportan, pero no se cambian automáticamente.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from pandas.api.types import is_object_dtype, is_string_dtype

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

RUTA_ACTUAL = Path.cwd().resolve()
RAIZ = next(
    ruta for ruta in [RUTA_ACTUAL, *RUTA_ACTUAL.parents]
    if (ruta / 'datos_procesados').exists()
)
RUTA_DATOS = RAIZ / 'datos_procesados'

RUTA_2024 = RUTA_DATOS / 'tabla_modelo_2024.csv'
RUTA_2025 = RUTA_DATOS / 'tabla_modelo_2025.csv'
RUTA_LIMPIA = RUTA_DATOS / 'tabla_modelo_2024_2025_limpia.csv'
RUTA_REPORTE = RUTA_DATOS / 'reporte_calidad_2024_2025.csv'
RUTA_REGISTRO = RUTA_DATOS / 'registro_limpieza_2024_2025.csv'
RUTA_RAROS = RUTA_DATOS / 'niveles_raros_2024_2025.csv'

for ruta in [RUTA_2024, RUTA_2025]:
    if not ruta.exists():
        raise FileNotFoundError(f'No se encontró el archivo: {ruta}')

In [ ]:
# Los identificadores permanecen como texto para evitar pérdida de formato.
TIPOS_ID = {'caseid': 'string', 'hhid': 'string', 'idx_nino': 'string'}

def cargar(ruta):
    return pd.read_csv(ruta, dtype=TIPOS_ID, low_memory=False)

tabla_2024 = cargar(RUTA_2024)
tabla_2025 = cargar(RUTA_2025)

if set(tabla_2024.columns) != set(tabla_2025.columns):
    raise ValueError('Las tablas no tienen las mismas columnas; revisar antes de limpiar.')

orden_columnas = tabla_2024.columns.tolist()
tabla_bruta = pd.concat(
    [tabla_2024[orden_columnas], tabla_2025[orden_columnas]],
    ignore_index=True
)

print(f'Tabla bruta: {tabla_bruta.shape[0]:,} filas × {tabla_bruta.shape[1]} columnas')
display(tabla_bruta['anio'].value_counts().sort_index().to_frame('n_ninos'))

## 1. Llaves, duplicados y literales de faltante

Primero se verifica la integridad de una fila por niño/año y se busca texto que aparente ser faltante.

In [ ]:
LLAVES = ['anio', 'caseid', 'idx_nino']
LITERALES_FALTANTE = {'', 'na', 'n/a', 'nan', 'none', 'null', '<na>'}

def texto_normalizado(serie):
    return serie.astype('string').str.strip().str.casefold()

resumen_llaves = pd.DataFrame([{
    'filas': len(tabla_bruta),
    'duplicados_anio_caseid_idx': tabla_bruta.duplicated(LLAVES).sum(),
    'anio_faltante': tabla_bruta['anio'].isna().sum(),
    'caseid_faltante': tabla_bruta['caseid'].isna().sum(),
    'idx_nino_faltante': tabla_bruta['idx_nino'].isna().sum(),
    'hhid_faltante': tabla_bruta['hhid'].isna().sum()
}])

literales_faltante = []
for columna in tabla_bruta.columns:
    if is_object_dtype(tabla_bruta[columna]) or is_string_dtype(tabla_bruta[columna]):
        n_literal = texto_normalizado(tabla_bruta[columna]).isin(LITERALES_FALTANTE).sum()
        if n_literal:
            literales_faltante.append({
                'variable': columna,
                'hallazgo': 'literal_de_faltante',
                'n_registros': int(n_literal),
                'porcentaje': n_literal / len(tabla_bruta)
            })

display(resumen_llaves)
display(pd.DataFrame(literales_faltante))

## 2. Limpieza segura

Se trabaja sobre una copia. Las fuentes `tabla_modelo_2024.csv` y `tabla_modelo_2025.csv` nunca se sobrescriben.

In [ ]:
tabla_limpia = tabla_bruta.copy()
registro_limpieza = []

# Normalizar espacios y literales de faltante exclusivamente en texto.
for columna in tabla_limpia.columns:
    if is_object_dtype(tabla_limpia[columna]) or is_string_dtype(tabla_limpia[columna]):
        original = tabla_limpia[columna].astype('string')
        sin_espacios = original.str.strip()
        espacios_corregidos = (original.notna() & original.ne(sin_espacios)).sum()
        es_faltante_textual = sin_espacios.str.casefold().isin(LITERALES_FALTANTE)
        faltantes_corregidos = es_faltante_textual.sum()

        tabla_limpia[columna] = sin_espacios.mask(es_faltante_textual, pd.NA)

        if espacios_corregidos or faltantes_corregidos:
            registro_limpieza.append({
                'variable': columna,
                'accion': 'recorte_espacios_y_faltantes_textuales',
                'n_espacios_corregidos': int(espacios_corregidos),
                'n_faltantes_textuales_convertidos': int(faltantes_corregidos)
            })

# 99.9 es el marcador de medición no válida/faltante en hemoglobina ajustada.
for columna in ['hb_ajustada_minsa_gdl', 'hb_ajustada_oms_gdl']:
    valores = pd.to_numeric(tabla_limpia[columna], errors='coerce')
    es_sentinel_999 = valores.eq(99.9)
    tabla_limpia[columna] = valores.mask(es_sentinel_999, np.nan)
    registro_limpieza.append({
        'variable': columna,
        'accion': 'sentinel_99_9_a_faltante',
        'n_espacios_corregidos': 0,
        'n_faltantes_textuales_convertidos': int(es_sentinel_999.sum())
    })

# Fechas creadas en 1634: se convierten a datetime; los formatos no válidos se registran.
COLUMNAS_FECHA = ['fecha_ultimo_cred', 'fecha_ultima_entrega_mmn']
for columna in COLUMNAS_FECHA:
    antes = tabla_limpia[columna].copy()
    despues = pd.to_datetime(antes, errors='coerce')
    fechas_invalidas = (antes.notna() & despues.isna()).sum()
    tabla_limpia[columna] = despues
    registro_limpieza.append({
        'variable': columna,
        'accion': 'conversion_a_fecha',
        'n_espacios_corregidos': 0,
        'n_faltantes_textuales_convertidos': 0,
        'n_fechas_invalidas_convertidas_a_nat': int(fechas_invalidas)
    })

registro_limpieza = pd.DataFrame(registro_limpieza)
registro_limpieza

## 3. Rangos, fechas y consistencia lógica

Los hallazgos de esta sección son alertas para revisar. No se reemplazan valores automáticamente.

In [ ]:
hallazgos = []

def registrar_hallazgo(variable, tipo, mascara, detalle):
    n = int(mascara.fillna(False).sum())
    if n:
        hallazgos.append({
            'variable': variable,
            'tipo_hallazgo': tipo,
            'n_registros': n,
            'porcentaje': n / len(tabla_limpia),
            'detalle': detalle
        })

# Rangos esperados. Un valor fuera de rango se reporta, no se borra.
RANGOS = {
    'edad_meses': (0, 59),
    'departamento_cod': (1, 25),
    'altitud_msnm': (-500, 7000),
    'quintil_riqueza': (1, 5),
    'controles_cred_acum': (0, 40),
    'n_cred_con_fecha': (0, 12),
    'n_entregas_mmn_registradas': (1, 6),
    'n_entregas_mmn_con_fecha': (0, 6),
    'edad_madre': (12, 60),
    'n_residentes_habituales': (1, 99),
    'n_menores_5_hogar': (0, 99),
    'n_ninos_6_17_hogar': (0, 99),
    'n_adultos_18_59_hogar': (0, 99),
    'n_adultos_mayores_hogar': (0, 99),
    'n_habitaciones_dormir': (0, 99),
    'personas_por_dormitorio': (0, 99),
    'hb_ajustada_minsa_gdl': (2, 25),
    'hb_ajustada_oms_gdl': (2, 25)
}

for variable, (minimo, maximo) in RANGOS.items():
    valores = pd.to_numeric(tabla_limpia[variable], errors='coerce')
    fuera_rango = valores.notna() & ~valores.between(minimo, maximo)
    registrar_hallazgo(
        variable, 'fuera_de_rango', fuera_rango,
        f'Rango esperado: {minimo} a {maximo}'
    )

# Variables con codificación binaria esperada.
BINARIAS_01 = [
    'recibio_jarabe', 'recibio_gotas', 'recibio_polvo', 'recibio_otra',
    'tomo7_jarabe', 'tomo7_gotas', 'tomo7_polvo', 'tomo7_otra',
    'tomo_7d', 'control_cred_alguno', 'anemia_minsa', 'anemia_oms',
    'razon_estomago', 'razon_diarrea', 'razon_estrenimiento',
    'razon_sabor', 'razon_dientes', 'razon_orientacion',
    'razon_inicio_suplementacion', 'razon_olvido', 'razon_otra'
]
for variable in BINARIAS_01:
    valores = pd.to_numeric(tabla_limpia[variable], errors='coerce')
    no_binario = valores.notna() & ~valores.isin([0, 1])
    registrar_hallazgo(variable, 'codigo_no_binario', no_binario, 'Se esperaba 0, 1 o faltante')

# En las preguntas sobre personal CRED, 8 significa 'no sabe'; es válido.
PERSONAL_CRED = [
    'cred_realizado_por_doctor', 'cred_realizado_por_obstetra',
    'cred_realizado_por_enfermera', 'cred_realizado_por_tecnico_enfermeria',
    'cred_realizado_por_promotor_salud', 'cred_realizado_por_otro'
]
for variable in PERSONAL_CRED:
    valores = pd.to_numeric(tabla_limpia[variable], errors='coerce')
    codigo_inesperado = valores.notna() & ~valores.isin([0, 1, 8])
    registrar_hallazgo(variable, 'codigo_inesperado_cred', codigo_inesperado, 'Se esperaba 0, 1, 8 o faltante')

# Fechas posteriores al año de la encuesta: alerta de posible error temporal.
for variable in COLUMNAS_FECHA:
    posterior = tabla_limpia[variable].notna() & (tabla_limpia[variable].dt.year > tabla_limpia['anio'])
    registrar_hallazgo(variable, 'fecha_posterior_a_encuesta', posterior, 'Año de fecha mayor al año ENDES')

# Relaciones lógicas entre variables.
registrar_hallazgo(
    'n_menores_5_hogar', 'inconsistencia_hogar',
    tabla_limpia['n_menores_5_hogar'] > tabla_limpia['n_residentes_habituales'],
    'Menores de 5 supera residentes habituales'
)

composicion_sin_menores = (
    tabla_limpia['n_ninos_6_17_hogar']
    + tabla_limpia['n_adultos_18_59_hogar']
    + tabla_limpia['n_adultos_mayores_hogar']
)
registrar_hallazgo(
    'composicion_hogar', 'inconsistencia_hogar',
    composicion_sin_menores + tabla_limpia['n_menores_5_hogar'] > tabla_limpia['n_residentes_habituales'],
    'Suma de grupos de edad supera residentes habituales'
)

registrar_hallazgo(
    'n_cred_con_fecha', 'inconsistencia_cred',
    tabla_limpia['n_cred_con_fecha'] > tabla_limpia['controles_cred_acum'],
    'Controles con fecha supera controles CRED acumulados'
)

registrar_hallazgo(
    'n_entregas_mmn_con_fecha', 'inconsistencia_mmn',
    tabla_limpia['n_entregas_mmn_registradas'].notna()
    & (tabla_limpia['n_entregas_mmn_con_fecha'] > tabla_limpia['n_entregas_mmn_registradas']),
    'Entregas con fecha supera entregas registradas'
)

relacion_hacinamiento = (
    tabla_limpia['n_residentes_habituales']
    / tabla_limpia['n_habitaciones_dormir']
)
comparables_hacinamiento = (
    tabla_limpia['n_habitaciones_dormir'].gt(0)
    & tabla_limpia['personas_por_dormitorio'].notna()
)
registrar_hallazgo(
    'personas_por_dormitorio', 'inconsistencia_calculo',
    comparables_hacinamiento
    & ~np.isclose(tabla_limpia['personas_por_dormitorio'], relacion_hacinamiento, equal_nan=True),
    'No coincide con residentes / habitaciones'
)

reporte_calidad = pd.DataFrame(hallazgos).sort_values(
    ['tipo_hallazgo', 'n_registros'], ascending=[True, False]
) if hallazgos else pd.DataFrame(
    columns=['variable', 'tipo_hallazgo', 'n_registros', 'porcentaje', 'detalle']
)

reporte_calidad

## 4. Cobertura, categorías raras y perfiles posteriores a limpieza

Los niveles raros no se eliminan aquí. Servirán para decidir agrupaciones o codificación durante la preparación de predictores.

In [ ]:
perfil_limpio = pd.DataFrame({
    'variable': tabla_limpia.columns,
    'tipo_limpio': tabla_limpia.dtypes.astype(str).values,
    'n_no_nulos': tabla_limpia.notna().sum().values,
    'porcentaje_nulos': tabla_limpia.isna().mean().values,
    'valores_distintos': [tabla_limpia[c].nunique(dropna=True) for c in tabla_limpia.columns]
})

niveles_raros = []
columnas_no_categoricas = {'caseid', 'hhid', 'idx_nino', *COLUMNAS_FECHA}
for columna in tabla_limpia.columns:
    if columna in columnas_no_categoricas:
        continue
    if is_object_dtype(tabla_limpia[columna]) or is_string_dtype(tabla_limpia[columna]):
        conteos = tabla_limpia[columna].value_counts(dropna=True)
        for valor, n in conteos[conteos <= 5].items():
            niveles_raros.append({
                'variable': columna,
                'nivel': str(valor),
                'n_registros': int(n),
                'porcentaje': n / len(tabla_limpia)
            })

niveles_raros = pd.DataFrame(niveles_raros)

# Una cobertura igual o menor a 5% impide usar una variable como predictora sin una decisión explícita.
cobertura_critica = perfil_limpio.loc[
    perfil_limpio['n_no_nulos'] / len(tabla_limpia) <= 0.05,
    ['variable', 'n_no_nulos', 'porcentaje_nulos']
].copy()
cobertura_critica['tipo_hallazgo'] = 'cobertura_critica'
cobertura_critica['n_registros'] = len(tabla_limpia) - cobertura_critica['n_no_nulos']
cobertura_critica['porcentaje'] = cobertura_critica['porcentaje_nulos']
cobertura_critica['detalle'] = 'Cobertura de datos menor o igual a 5%'
cobertura_critica = cobertura_critica[[
    'variable', 'tipo_hallazgo', 'n_registros', 'porcentaje', 'detalle'
]]
reporte_calidad = pd.concat([reporte_calidad, cobertura_critica], ignore_index=True)


display(perfil_limpio.sort_values('porcentaje_nulos', ascending=False).head(20))
display(niveles_raros.sort_values(['variable', 'n_registros']) if not niveles_raros.empty else niveles_raros)

## 5. Guardar resultados

Se guarda una copia limpia unificada y tres reportes. Las tablas por año originales no se sobrescriben.

In [ ]:
tabla_limpia.to_csv(RUTA_LIMPIA, index=False, encoding='utf-8-sig')
reporte_calidad.to_csv(RUTA_REPORTE, index=False, encoding='utf-8-sig')
registro_limpieza.to_csv(RUTA_REGISTRO, index=False, encoding='utf-8-sig')
niveles_raros.to_csv(RUTA_RAROS, index=False, encoding='utf-8-sig')

print(f'Tabla limpia: {RUTA_LIMPIA.name} — {tabla_limpia.shape[0]:,} filas × {tabla_limpia.shape[1]} columnas')
print(f'Reporte de calidad: {RUTA_REPORTE.name} — {len(reporte_calidad):,} hallazgos')
print(f'Registro de limpieza: {RUTA_REGISTRO.name} — {len(registro_limpieza):,} acciones')
print(f'Niveles raros: {RUTA_RAROS.name} — {len(niveles_raros):,} niveles')